<a href="https://colab.research.google.com/github/Ghanasree-S/LXNet/blob/main/MIP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os
os.environ["KAGGLE_API_TOKEN"] = "KGAT_986453a136a328fe9883366e9e11f223"

In [14]:
!pip install -q kaggle
!kaggle datasets download -d fernando2rad/x-ray-lung-diseases-images-9-classes -p /content/data --unzip

Dataset URL: https://www.kaggle.com/datasets/fernando2rad/x-ray-lung-diseases-images-9-classes
License(s): apache-2.0
100% 182M/182M [00:01<00:00, 115MB/s]



In [15]:
import os
DATA_ROOT = "/content/data"
print(sorted(os.listdir(DATA_ROOT)))

['00 Anatomia Normal', '01 Processos Inflamatórios Pulmonares (Pneumonia)', '02 Maior Densidade (Derrame Pleural, Consolidação Atelectasica, Hidrotorax, Empiema)', '03 Menor Densidade (Pneumotorax, Pneumomediastino, Pneumoperitonio)', '04 Doenças Pulmonares Obstrutivas (Enfisema, Broncopneumonia, Bronquiectasia, Embolia)', '05 Doenças Infecciosas Degenerativas (Tuberculose, Sarcoidose, Proteinose, Fibrose)', '06 Lesões Encapsuladas (Abscessos, Nódulos, Cistos, Massas Tumorais, Metastases)', '07 Alterações de Mediastino (Pericardite, Malformações Arteriovenosas, Linfonodomegalias)', '08 Alterações do Tórax (Atelectasias, Malformações, Agenesia, Hipoplasias)']


In [16]:
for root, dirs, files in os.walk(DATA_ROOT):
    print(root, dirs[:12], len(files))
    if root.count(os.sep) - DATA_ROOT.count(os.sep) > 2:
        break

/content/data ['03 Menor Densidade (Pneumotorax, Pneumomediastino, Pneumoperitonio)', '08 Alterações do Tórax (Atelectasias, Malformações, Agenesia, Hipoplasias)', '01 Processos Inflamatórios Pulmonares (Pneumonia)', '06 Lesões Encapsuladas (Abscessos, Nódulos, Cistos, Massas Tumorais, Metastases)', '02 Maior Densidade (Derrame Pleural, Consolidação Atelectasica, Hidrotorax, Empiema)', '04 Doenças Pulmonares Obstrutivas (Enfisema, Broncopneumonia, Bronquiectasia, Embolia)', '00 Anatomia Normal', '05 Doenças Infecciosas Degenerativas (Tuberculose, Sarcoidose, Proteinose, Fibrose)', '07 Alterações de Mediastino (Pericardite, Malformações Arteriovenosas, Linfonodomegalias)'] 0
/content/data/03 Menor Densidade (Pneumotorax, Pneumomediastino, Pneumoperitonio) [] 629
/content/data/08 Alterações do Tórax (Atelectasias, Malformações, Agenesia, Hipoplasias) [] 544
/content/data/01 Processos Inflamatórios Pulmonares (Pneumonia) [] 1060
/content/data/06 Lesões Encapsuladas (Abscessos, Nódulos, Ci

In [17]:
!git clone https://github.com/Ghanasree-S/LXNet.git
%cd LXNet
!pip install -q opencv-python-headless scikit-learn scipy matplotlib pandas lime scikit-image
!pip install -q -e . --no-deps

Cloning into 'LXNet'...
remote: Enumerating objects: 59, done.
remote: Counting objects: 100% (59/59), done.
remote: Compressing objects: 100% (57/57), done.
remote: Total 59 (delta 0), reused 59 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (59/59), 7.94 MiB | 12.49 MiB/s, done.
/content/LXNet/LXNet
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
ERROR: Package 'lxnet' requires a different Python: 3.12.13 not in '<3.11,>=3.10'


In [18]:
import sys
sys.path.insert(0, "/content/LXNet")

In [19]:
import lxnet
from lxnet import train
print("ok")

ok


In [20]:
import tensorflow as tf
print(tf.__version__)
print(tf.config.list_physical_devices("GPU"))

2.20.0
[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [21]:
!python -m lxnet.train --data-root /content/data --out-dir runs/smoke \
  --split-mode random --limit 200 --epochs 2 --folds 2 --models LXNet

17:34:10 INFO GPUs visible: ['/physical_device:GPU:0']
17:34:11 INFO indexed 6743 images
17:34:11 WARNING dropping 2 copies of an image labelled as ['01 Processos Inflamatórios Pulmonares (Pneumonia)', '03 Menor Densidade (Pneumotorax, Pneumomediastino, Pneumoperitonio)']
17:34:11 INFO dedupe: kept 6657 images; dropped 84 exact duplicates and 1 cross-class conflicts
17:34:17 INFO preprocessed 1000/6657 images
17:34:22 INFO preprocessed 2000/6657 images
17:34:26 INFO preprocessed 3000/6657 images
17:34:31 INFO preprocessed 4000/6657 images
17:34:35 INFO preprocessed 5000/6657 images
17:34:39 INFO preprocessed 6000/6657 images
17:35:11 INFO wrote preprocessed cache to runs/smoke/preprocessed.npz
17:35:11 INFO class distribution: {'Chest Changes': 16, 'Degenerative Infectious': 18, 'Encapsulated Lesions': 19, 'Higher Density': 20, 'Lower Density': 18, 'Mediastinal Changes': 18, 'Normal': 41, 'Obstructive Pulmonary': 19, 'Pneumonia': 31}
17:35:11 INFO split mode: random
17:35:11 INFO split

In [30]:
!nohup python -m lxnet.train --data-root /content/data --out-dir runs/paper_replication \
  --split-mode random --epochs 40 --batch-size 32 --folds 5 --seed 42 \
  --cv-models LXNet DenseNet201 ResNet50V2 InceptionV3 > train.log 2>&1 &

In [41]:
!tail -30 train.log

264/264 - 39s - 146ms/step - accuracy: 0.9561 - loss: 0.1798 - val_accuracy: 0.9429 - val_loss: 0.1992 - learning_rate: 1.0000e-04
Epoch 14/40
264/264 - 39s - 147ms/step - accuracy: 0.9666 - loss: 0.1593 - val_accuracy: 0.9489 - val_loss: 0.1843 - learning_rate: 1.0000e-04
Epoch 15/40
264/264 - 39s - 148ms/step - accuracy: 0.9695 - loss: 0.1426 - val_accuracy: 0.9499 - val_loss: 0.1824 - learning_rate: 1.0000e-04
Epoch 16/40
264/264 - 39s - 146ms/step - accuracy: 0.9712 - loss: 0.1292 - val_accuracy: 0.9530 - val_loss: 0.1627 - learning_rate: 1.0000e-04
Epoch 17/40
264/264 - 37s - 141ms/step - accuracy: 0.9715 - loss: 0.1205 - val_accuracy: 0.9520 - val_loss: 0.1634 - learning_rate: 1.0000e-04
Epoch 18/40
264/264 - 39s - 147ms/step - accuracy: 0.9765 - loss: 0.1089 - val_accuracy: 0.9580 - val_loss: 0.1410 - learning_rate: 1.0000e-04
Epoch 19/40
264/264 - 37s - 139ms/step - accuracy: 0.9774 - loss: 0.1013 - val_accuracy: 0.9570 - val_loss: 0.1428 - learning_rate: 1.0000e-04
Epoch 20/40